# Stage 3 - clean-data baselines (Random Forest and 1D-CNN)

Train both models on the prepared arrays and evaluate on the held-out test set. Read macro-F1, the per-class report and the confusion matrix - not overall accuracy. The CNN uses balanced class weights only where the dataset config asks for it.

In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically: every step below runs the SAME
# code for both. The only dataset-specific code in the project is each
# dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Setup - device and a small loader

In [2]:
import torch, joblib
from sklearn.utils.class_weight import compute_class_weight
from adversec.models import build_random_forest, CNN1D, train_cnn
from adversec.evaluation import evaluate_model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEVICE)

def load_arrays(name):
    a = np.load(config.PROCESSED_DIR / f'{name}_stage2_arrays.npz')
    classes = list(joblib.load(config.PROCESSED_DIR / f'{name}_label_encoder.joblib').classes_)
    return a['X_train'], a['y_train'], a['X_test'], a['y_test'], classes

device: cuda


## Random Forest baseline (both datasets)

In [3]:
rf_models, rf_metrics, test_meta = {}, {}, {}
for name in DATASETS:
    Xtr, ytr, Xte, yte, classes = load_arrays(name)
    rf = build_random_forest(); rf.fit(Xtr, ytr); rf_models[name] = rf
    rf_metrics[name] = evaluate_model(yte, rf.predict(Xte), classes, model_name=f'{name} - Random Forest')
    test_meta[name] = {'classes': classes, 'test_size': int(len(yte))}


    ciciov2024 - Random Forest
    Accuracy    : 0.9972
    Macro-F1    : 0.7761

   Per-class report:
                         precision    recall  f1-score   support

                    DoS       1.00      0.75      0.86         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       0.00      0.00      0.00         1
           spoofing-RPM       0.67      1.00      0.80         2
         spoofing-SPEED       1.00      1.00      1.00         1
spoofing-STEERING_WHEEL       1.00      1.00      1.00         1

               accuracy                           1.00       718
              macro avg       0.78      0.79      0.78       718
           weighted avg       1.00      1.00      1.00       718

     Confusion matrix (rows=true, cols=pred):
[[  3   0   0   1   0   0]
 [  0 709   0   0   0   0]
 [  0   1   0   0   0   0]
 [  0   0   0   2   0   0]
 [  0   0   0   0   1   0]
 [  0   0   0   0   0   1]]

    road - Random Forest
    Accu

## 1D-CNN baseline (both datasets)
Watch the per-epoch loss, then read the same metric set as the RF.

In [4]:
cnn_models, cnn_metrics = {}, {}
for name in DATASETS:
    Xtr, ytr, Xte, yte, classes = load_arrays(name)
    cfg = config.load_dataset_config(name)
    cw = None
    if cfg.get('cnn_class_weights'):
        w = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
        cw = torch.tensor(w, dtype=torch.float32, device=DEVICE)
    print(f'\n=== training CNN: {name} (class weights: {bool(cw is not None)}) ===')
    cnn = CNN1D(n_features=Xtr.shape[1], n_classes=len(classes))
    cnn = train_cnn(cnn, Xtr, ytr, n_epochs=50, device=DEVICE, class_weights=cw)
    cnn_models[name] = cnn
    cnn.eval()
    with torch.no_grad():
        pred = cnn(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).argmax(1).cpu().numpy()
    cnn_metrics[name] = evaluate_model(yte, pred, classes, model_name=f'{name} - 1D-CNN')


=== training CNN: ciciov2024 (class weights: True) ===
    epoch   1/50     loss 1.6527
    epoch   5/50     loss 0.1676
    epoch  10/50     loss 0.0124
    epoch  15/50     loss 0.0031
    epoch  20/50     loss 0.0021
    epoch  25/50     loss 0.0014
    epoch  30/50     loss 0.0011
    epoch  35/50     loss 0.0009
    epoch  40/50     loss 0.0009
    epoch  45/50     loss 0.0006
    epoch  50/50     loss 0.0007

    ciciov2024 - 1D-CNN
    Accuracy    : 0.9930
    Macro-F1    : 0.6606

   Per-class report:
                         precision    recall  f1-score   support

                    DoS       0.67      1.00      0.80         4
                 benign       1.00      1.00      1.00       709
           spoofing-GAS       1.00      1.00      1.00         1
           spoofing-RPM       0.50      0.50      0.50         2
         spoofing-SPEED       0.50      1.00      0.67         1
spoofing-STEERING_WHEEL       0.00      0.00      0.00         1

               accuracy    

## Save baseline metrics (for report writing + later notebooks)
Writes `results/<name>_baseline_metrics.json` — the same artifact `adversec baseline` produces: accuracy, macro-F1 and confusion matrix for both models.

In [5]:
import json
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    report = {
        'dataset': name,
        'device': DEVICE,
        'n_classes': len(test_meta[name]['classes']),
        'class_names': test_meta[name]['classes'],
        'test_size': test_meta[name]['test_size'],
        'random_forest': {k: rf_metrics[name][k] for k in ('accuracy', 'macro_f1', 'confusion_matrix')},
        'cnn_1d': {k: cnn_metrics[name][k] for k in ('accuracy', 'macro_f1', 'confusion_matrix')},
    }
    path = config.RESULTS_DIR / f'{name}_baseline_metrics.json'
    path.write_text(json.dumps(report, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_baseline_metrics.json
saved -> /home/koala/lab/adversec/results/road_baseline_metrics.json
